# FlashAttention/IO-aware Attention 실습 - Online Softmax Block Merge

- Tutorial ID: `mod-flashattention-lab`
- Tutorial: FlashAttention/IO-aware Attention 실습
- Section ID: `flash-1`
- Section: Online Softmax Block Merge


In [ ]:
# ============================================================
# 코드 읽는 법 — Online Softmax Block Merge
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) logit이 확률분포로 바뀌는 과정과 temperature/top-k/top-p의 효과 관찰
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import numpy as np
np.random.seed(1)
scores = np.random.randn(12) * 3
# exact softmax
exact = np.exp(scores - scores.max()); exact /= exact.sum()
# online over blocks
m = -np.inf; l = 0.0; weighted = np.zeros_like(scores)
start = 0
for block in np.array_split(scores, 3):
    mb = block.max(); eb = np.exp(block - mb); lb = eb.sum()
    new_m = max(m, mb)
    # 이전 누적분과 새 블록을 같은 scale로 재조정
    weighted[:start] *= np.exp(m - new_m)
    weighted[start:start+len(block)] = eb * np.exp(mb - new_m)
    l = l * np.exp(m - new_m) + lb * np.exp(mb - new_m)
    m = new_m; start += len(block)
online = weighted / l
print('max abs error:', np.max(np.abs(exact-online)))
print('exact :', np.round(exact, 5))
print('online:', np.round(online, 5))